```2026/09/10 09:48```

# Optimization characterization for `CAXSim`

This notebook aims to study and guide the application of optimization algorithms
to the Carcará-X beamline simulation. Aside from the optimization method, there 
are multiple aspects of the optimization procedure that require careful consideration:

- A. Objective function choice

- B. Measurement procedure

- C. Degrees of freedom / # of knobs

- D. Optimization method parameters

## A. objective functions

### 1. Simple size/intensity ratio

A function of the form: $f_1(\sigma_x, \sigma_y, I) = \dfrac{\sigma_x \sigma_y}{I}$

This form is very simple, but it has a few drawbacks:

- Intensity is hard to define/compare between simulation and real beamline;
- The two sigmas have different effective wheights, as the beam is usually 
stretched horizontally.

### 2. Normalized pure geometric

A function of the form: $f_2(\sigma_x, \sigma_y) = \dfrac{\sigma_x}{ \sigma_x^0} 
                                                    + \dfrac{\sigma_x}{ \sigma_x^0}$

where the normalization factors $\sigma_{x/y}^0$ are the values given by the 
Sirius model for the electron beam sizes at the site of CAX.

This function balances out the weight of the two sigmas but doesn't take intensity
into account.

### 3. Pure intensity

A function of the form: $f_3(I) = -\dfrac{I}{I_0}$

where $I_0$ is a normalization term for intensity, to be identified.

In theory, if intensity is well defined and the whole parameter space is searched,
the optimizer should find the tightest beam where intensity is maximized.

### 4. Geometric + intensity

A function of the form: $f_4(\sigma_x, \sigma_y, I) = f_2(\sigma_x, \sigma_y)
                                                    + f_3(I)$

This further restricts the objective, but takes into account both geometrical and
intensity features.

### 5. Astigmatism

A function of the form $f_5(d) = \dfrac{|d|}{\Delta}$

where $d$ represents the difference in foci positions $d = f_y - f_x$. The 
normalization parameter $\Delta$ should be of the order of magnitude of the 
variations in astigmatism.

The drawback in this case is the fact that astigmatism acquisition is slow and 
expensive since it relies on a sweep of the DVF $z$ position. On the beamline
it appears that the DVF's rails do not extend far enough to really capture the 
positions of both foci, and as such, astigmatism acquisition strategies are also
important to consider.

### 6. Geometric + astigmatism / All

Finally, we combine either $f_2$, $f_3$ and $f_4$ with $f_5$. 

## B. Acquisition Method (DVF placement)

### 1. DVF at fixed original position ($z=0$)

Performing acquisitions with the DVF device's position fixed at $z=0$ (in relation
to it's own coordinate system) is the most natural acquisition method in the real
beamline. However, as the beam foci vary with incidence angle, optimizing a particular
objective function with observable values taken at $z=0$ does not necessarily produce
an optimal beam image at the focus position.

The advantage of this type of acquisition is how cheap it is, not prompting us to
move the DVF at all.

### 2. DVF at focus ($z = \bar f$)

Defining a focus position is tricky in itself as the M1 toroidal mirror introduces 
an astigmatism effect due to its two different radii of curvature. An option is 
to define $\bar f \equiv (f_x +f_y)/2$ as the *avarege focus* of the system. An 
optimization that matches the foci will reduce the astigmatism and determine the 
true focus as $\bar f$ itself.

The natural advantage of this methos is the fact that it takes into account the
foci matching objective and will probably optimize faster as multiple observables
will be optimal in the true matched focus of the optical system.

The clear disadvantage is the fact that one needs to know $f_x$ and $f_y$ to 
compute $\bar f$, which requires a caustic scan and is inherently expensive and 
potentially noisy (or sensitive to motor position uncertainties). If we're able 
to obtain good results using the previous acquisiton method the optimization will
be greatly accelerated.

---

First, we'll set up methods to characterize the different objective functions.
Starting with just two knobs, the mirror's translation in x and the y rotation
we'll first measure sigmas, intensity and astigmatism for every combination of 
TX and RY. We'll also save acquisitions taken with the DVF placed exactly at the
midpoint of the foci $\bar f$.



In [ ]:
# Imports

import os
import sys

import matplotlib.pyplot as plt

# from Shadow import histo
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from shadowpy.si_beamlines import CAXSim
from shadowpy.utils import Image, save_image

sys.path.append(os.path.expanduser("~/repos/cax-scripts"))
from caxscripts.image_statistics import Histogram2DAnalyzer

def add_minimum_band_legend(ax, x, y, values, fraction=0.005, legend=False):
    """Highlight and label the region within `fraction` of the minimum."""
    minimum = np.nanmin(values)
    level = minimum + fraction * np.ptp(values)

    ax.contourf(
        x, y, values,
        levels=[minimum, level],
        colors=["red"],
        alpha=0.3,
    )

    proxy = Patch(
        facecolor="red",
        alpha=0.3,
        label=f"{fraction:.1%} above minimum",
    )

    handles, labels = ax.get_legend_handles_labels()
    if legend:
        ax.legend(
            handles=[*handles, proxy],
            loc="lower right",
            fontsize=10,
            frameon=True,
            facecolor="white",
            edgecolor="black",
        )

In [ ]:
def _astigmatism_acquisition(cax: CAXSim):

    # Initial search

    low_lim = -1000
    hgh_lim =  1000

    done = False

    while not done:
        distances, fwhm_x, fwhm_y = \
            cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam.duplicate(),
                                        s_range=(low_lim, hgh_lim),
                                        n_points=50, 
                                        max_workers=10)
        # prior focal lengths
        fx = distances[np.nanargmin(fwhm_x)]
        fy = distances[np.nanargmin(fwhm_y)]

        print(fx, fy)

        done = True

        # check if the search was enought otherwise repeat
        if low_lim in [fx, fy]:
            low_lim -= 100
            done = False

        if hgh_lim in [fx, fy]:
            hgh_lim += 100
            done = False

    # Finer search
    distances, fwhm_x, fwhm_y = \
    cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam.duplicate(),
                                s_range=(np.min([fx, fy])-50, np.max([fx, fy])+50),
                                n_points=100, 
                                max_workers=10)

    # actual focal lengths
    fx = distances[np.nanargmin(fwhm_x)]
    fy = distances[np.nanargmin(fwhm_y)]
    print(fx, fy)

    f = np.mean([fx, fy])
    retr_beam = cax.dvf_B1.beam.duplicate()
    retr_beam.retrace(f)
    img = save_image(cax.dvf_B1, retr_beam)
    astig = fy - fx

    return distances, fwhm_x, fwhm_y, fx, fy, f, astig, img 


In [ ]:
def display_image(image: Image):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    hist = image.histogram

    p = ax.pcolormesh(image.bin_h_edges, image.bin_v_edges, hist.T)
    fig.colorbar(p, ax=ax, label="count")

    ax.set_xlabel('x [mm]')
    ax.set_ylabel('y [mm]')

    return fig, ax

In [ ]:
cax = CAXSim(total_rays=500000, image_size=(100, 100))

In [ ]:
cax.mirror.tx = -2
fig, ax = display_image(cax.dvf_B1.image)


First, we need to define intensity in a precise way. As we are using raytracing simulation, the final
image is a histogram where each bin contains a certain ammount of rays. A way to define intensity is
to count how many rays lie inside a certain area of the beam, that should be proportional to energy 
divided by area.

In [ ]:
def ana_from_image(image: Image):

    ana = Histogram2DAnalyzer(
        img=image.histogram,
        x_bin_edges=image.bin_h_edges*1000,
        y_bin_edges=image.bin_v_edges*1000
    )

    return ana

In [ ]:
def peak(image: Image, droi=3):

    ana = ana_from_image(image)

    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_fitting)

    return ana.hprm_fitting['amplitude']


In [ ]:
# droi = 3

# cx, cy = img.fitx.roi_mean, img.fity.roi_mean
# cx, cy = int(cx), int(cy)
# peak = np.mean(img.data.T[cx - droi:cx + droi + 1, cy - droi:cy + droi + 1])

# peak


In [ ]:
cax.mirror.ry = 0

In [ ]:
peaks = []

ry_array = np.linspace(-.1, .1, 61)

for _ry in ry_array:
    cax.mirror.ry = _ry

    peaks.append(peak(cax.dvf_B1.image, droi=5))

In [ ]:
plt.plot(ry_array, peaks)
# plt.plot(ry_vals[0], intensity_vals[:, 40]/3000)
plt.show()

The acquisition was made using a fine grid of TX and RY values and stored as a csv. Interpolation was
used to acquire some intermediate values which were not stored in the first run. We will import the
data into pandas dataframes here:

In [ ]:
# df_at_dvf = pd.read_csv("knobspace_data/df_at_dvf_fine_interpolated.csv")
# df_at_focus = pd.read_csv("knobspace_data/df_at_focus_fine_interpolated.csv")
df_at_dvf = pd.read_csv("knobspace_data/df_at_dvf_v2.csv")
df_at_focus = pd.read_csv("knobspace_data/df_at_focus_v2.csv")

In [ ]:
df_at_dvf

In [ ]:
new_data = pd.read_csv('20261028_data.csv')

In [ ]:
# append just the 'astigmatism' from the original df_at_dvf to the new_data

new_data['astigmatism'] = df_at_dvf['astigmatism']

In [ ]:
new_data = pd.read_csv('new_data.csv')

In [ ]:
new_data

Extracting the values from the dataframes

In [ ]:
new_data.to_csv('knobspace_data/df_at_dvf_v6.csv', index=False)

In [ ]:
df_at_dvf = pd.read_csv("knobspace_data/df_at_dvf_v6.csv")

In [ ]:
# N = len(df_at_dvf)
# n = np.sqrt(N)
n = 81
tx_vals = np.asarray(df_at_dvf['TX']).reshape(int(n), int(n))
ry_vals = np.asarray(df_at_dvf['RY']).reshape(int(n), int(n))

intensity_vals = np.asarray(df_at_dvf['Intensity']).reshape(int(n), int(n))
sigmax_vals = np.asarray(df_at_dvf['sigma_x']).reshape(int(n), int(n))
sigmay_vals = np.asarray(df_at_dvf['sigma_y']).reshape(int(n), int(n))

astig_vals = np.asarray(df_at_dvf['astigmatism']).reshape(int(n), int(n))

TX, RY = np.meshgrid(tx_vals, ry_vals)

Let's have a look at the observables individually, performing the necessary sign 
inversions, applying `abs` for the astigmatism and changing sigma's units to $\mu$ m

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(17, 11))

axs = axs.flatten()

fig.suptitle("Observables at DVF position ($z = 0$)", fontsize=16)


obs = [-intensity_vals, np.abs(astig_vals), 1000*sigmax_vals, 1000*sigmay_vals]
obs_lbl = ["$-I$ [count]", "$|d|$ [mm]", "$\\sigma_x$ [$\\mu$ m]", "$\\sigma_y$ [$\\mu$ m]"]
obs_names = ["negative Intensity", "Astigmatism", "Sigma x", "Sigma y"]

for i in range(4):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(
        tx_vals, ry_vals, obs[i],
        levels=15,
        colors="white",
        linewidths=0.2,
    )
    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], legend=False)
    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Minimum at ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



Let's now look at the proposed objective functions. The normalization constants will be as follow:

$I_0 \equiv $ half of `total_rays = 50 000`;

$\Delta = 500 ~mm$, the total reach of CAX's DVF z translation;

$\sigma_x^0 = 19.9 ~\mu m$;

$\sigma_y^0 = 11.5 ~\mu m$;

Before defining the functions, let's check the normalized observables:

In [ ]:
# I0 = 0.5*cax.total_rays
I0 = 10
Delta = 500
sigmax_0 = 19.9
sigmay_0 = 11.5

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(17, 11))

axs = axs.flatten()

fig.suptitle("Normalized Observables at DVF ($z=0$)", fontsize=16)


obs = [-intensity_vals/I0, np.abs(astig_vals)/Delta, 1000*sigmax_vals/sigmax_0, 1000*sigmay_vals/sigmay_0]
obs_lbl = ["$-I/I_0$", "$|d|/\\Delta$", "$\\sigma_x/\\sigma_x^0$", "$\\sigma_y/\\sigma_y^0$"]
obs_names = ["normalized Intensity", "normalized Astigmatism", "normalized Sigma x", "normalized Sigma y"]

for i in range(4):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(tx_vals, ry_vals, obs[i], levels=15, colors='white', linewidths=0.2)

    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], legend=False)
    
    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Minimum at ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



Next we compute the 8 objectives discussed at the start of this section check their minima:

In [ ]:
f1_dvf = (1000)**2*sigmax_vals*sigmay_vals/intensity_vals
f2_dvf = 1000*sigmax_vals/sigmax_0 + 1000*sigmay_vals/sigmay_0
f3_dvf = -intensity_vals/I0
f4_dvf = f2_dvf + f3_dvf
f5_dvf = np.abs(astig_vals)/Delta
f6_dvf = f2_dvf + f5_dvf
f7_dvf = f3_dvf + f5_dvf
f8_dvf = f4_dvf + f5_dvf

In [ ]:
%matplotlib qt

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(17, 24))

axs = axs.flatten()

fig.suptitle("Objectives at DVF ($z=0$)", fontsize=16)

obs = [f1_dvf, f2_dvf, f3_dvf, f4_dvf, f5_dvf, f6_dvf, f7_dvf, f8_dvf]
obs_lbl = ["$\\sigma_x\\sigma_y/I$", 
           "$\\sigma_x/\\sigma_x^0 + \\sigma_y/\\sigma_y^0$", 
           "$-I/I_0$", 
           "$f_2(\\sigma_x, \\sigma_y) + f_3(I)$",
           "$|d|/\\Delta$",
           "$f_2(\\sigma_x, \\sigma_y) + f_5(d)$",
           "$f_3(I) + f_5(d)$",
           "$f_4(\\sigma_x, \\sigma_y, I) + f_5(d)$"]
obs_names = ["Mixed geometric intensity $f_1(\\sigma_x, \\sigma_y, I)$", 
             "Geometric $f_2(\\sigma_x, \\sigma_y)$", 
             "Intensity $f_3(I)$",
             "Geometric and intensity $f_4(\\sigma_x, \\sigma_y, I)$", 
             "Astigmatism $f_5(d)$",
             "Geometric and astigmatism $f_6(\\sigma_x, \\sigma_y, d)$",
             "Intensity and astigmatism $f_7(I, d)$",
             "Geometric, intensity and astigmatism $f_8(\\sigma_x, \\sigma_y, I, d)$"]

from matplotlib.colors import LogNorm

for i in range(8):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')#, vmax=30)
    # p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto', norm=LogNorm(vmin=obs[i][obs[i] > 0].min(), vmax=obs[i][obs[i] > 0].max()))#, vmax=30)
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(tx_vals, ry_vals, obs[i], levels=15, colors='white', linewidths=0.2)

    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], fraction=0.01, legend=False)

    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Min @ ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



Let's check the minimum of $f_8(\sigma_x, \sigma_y, I, d)$ which is the most 
restrictive objeective. We'll see first what is the beam size when projected onto 
the DVF and then the beam profile at the focus position:

In [ ]:
cax.mirror.offset = [0., 0., 0.]

In [ ]:
argmin = np.unravel_index(f2_dvf.argmin(), f2_dvf.shape)
# argmin

tx_s, ry_s = tx_vals[argmin], ry_vals[argmin]

cax.mirror.tx = tx_s
cax.mirror.ry = ry_s

display_image(cax.dvf_B1.image)
plt.show()

img = Image2D_Fit(data=cax.dvf_B1.image.histogram.T, use_svd4theta=True)

sig_x = img.fitx.roi_sigma
Del_x = np.diff(cax.dvf_B1.image.bin_h_edges)[0]*1000   # Δx in μm
sig_x *= Del_x

sig_y = img.fity.roi_sigma
Del_y = np.diff(cax.dvf_B1.image.bin_v_edges)[0]*1000   # Δy in μm
sig_y *= Del_y


print(f"Sigma x at DVF: {sig_x:.2f} μm")
print(f"Sigma y at DVF: {sig_y:.2f} μm")

That doesn't look like it is optimized, the beam size is `21.76 μm` by `13.08 μm`.
But we must remember that the focus is actually behind the DVF by some 10 cm. 
Let's look at a caustic scan, check if the astigmatism is actually low and
evaluate the image at $f = (f_x + f_y)/2$:

In [ ]:
distances, fwhm_x, fwhm_y, fx, fy, f, astig, image = _astigmatism_acquisition(cax)



plt.figure(figsize=(12, 6))
plt.plot(distances, 1000*fwhm_x, marker='o', markersize=5, label="Sigma X [μm]")
plt.plot(distances, 1000*fwhm_y, marker='o', markersize=5, label="Sigma Y [μm]")
plt.vlines(fx, 0, 1000*np.nanmin(fwhm_x), color='blue', linestyle='--', label=f"Min Sigma X: {1000*np.nanmin(fwhm_x):.2f} μm at {distances[np.nanargmin(fwhm_x)]:.2f} mm")
plt.vlines(fy, 0, 1000*np.nanmin(fwhm_y), color='orange', linestyle='--', label=f"Min Sigma Y: {1000*np.nanmin(fwhm_y):.2f} μm at {distances[np.nanargmin(fwhm_y)]:.2f} mm")
plt.vlines(f, 0, 1000*np.nanmax(fwhm_x), color='black', linestyle='solid', label=f"Midpoint of foci at {f:.2f} mm")
plt.xlabel('Distance from screen (mm)')
plt.ylabel('Sigma (μm)')
plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
plt.legend()
plt.grid()
plt.show()

display_image(image)
plt.show()

img = Image2D_Fit(data=image.histogram.T, use_svd4theta=True)

sig_x = img.fitx.roi_sigma
Del_x = np.diff(image.bin_h_edges)[0]*1000   # Δx in μm
sig_x *= Del_x

sig_y = img.fity.roi_sigma
Del_y = np.diff(image.bin_v_edges)[0]*1000   # Δy in μm
sig_y *= Del_y


print(f"Sigma x at DVF: {sig_x:.2f} μm")
print(f"Sigma y at DVF: {sig_y:.2f} μm")

Now that looks like a proper minimum: the spatial sizes are `19.88 μm` by `11.37 μm` which are actually
in the range of the nominal values considering the associated uncertainty, namely:

- $\sigma_x = (19.9 \pm 0.2) ~\mu m$
- $\sigma_y = (11.5 \pm 1.2) ~\mu m$

An important question to answear is weather optimizing the beam with no astigmatism information
can actually match the foci as a result. This is useful as astigmatism acquisition is expensive and
may be prohibitive in the real beamline. For instance, $f_2(\sigma_x, \sigma_y)$ is a purely geometric
objective but the above grids show it's minimum as being on the edge of our sampled space. Let's 
compare the objective minimum and its value at the $TX^*$ and $RY^*$ positions obtained from the more
detailed $f_8$:

In [ ]:
f2_argmin = np.argmin(f2_dvf)
f8_argmin = np.argmin(f8_dvf)
    
print(f"f2's range: ({np.min(f2_dvf):.2f} — {np.max(f2_dvf):.2f}) → {np.ptp(f2_dvf):.3f}")
print(f"Minimum of f2: {np.min(f2_dvf):.2f}")
print(f"f2 evaluated at f8's optimun location: {f2_dvf[argmin]:.2f}")
delt = f2_dvf[argmin] - np.min(f2_dvf)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f2_dvf)*100:.2f}% of full range")
print()
print(f"f8's range: ({np.min(f8_dvf):.2f} — {np.max(f8_dvf):.2f}) → {np.ptp(f8_dvf):.3f}")
print(f"Minimum of f8: {np.min(f8_dvf):.2f}")
print(f"f8 evaluated at f2's optimun location: {f8_dvf[np.unravel_index(f2_argmin, f2_dvf.shape)]:.2f}")
delt = f8_dvf[np.unravel_index(f2_argmin, f2_dvf.shape)] - np.min(f8_dvf)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f8_dvf)*100:.2f}% of full range")
print("\n Beam sizes at DVF position:")

print(f"\nAt f2's minimum: \n \t {df_at_dvf['sigma_x'][f2_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f2_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_dvf['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f8_argmin]*1000:.2f} μm")

print("\n Beam sizes at focus position:")

print(f"\nAt f2's minimum: \n \t {df_at_focus['sigma_x'][f2_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f2_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_focus['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f8_argmin]*1000:.2f} μm")


print(f"\n Astigmatism at f2's minimum: {df_at_dvf['astigmatism'][f2_argmin]:.2f} mm")

Unfortunatelly the pure geometric optimization doesn't match the foci automatically, yelding an 
astigmatism of ≈ 200 mm. We see a tradeoff between sigmas at the DVF position for $f_2$ and $f_8$ 
minima. At the focus the beam sizes are comparable, but the $f_8$ objective still wins.

Intensity information, although harder to define across simulation and real beamline, is still easily
obtainable without expensive scan procedures, let's now compare $f_8$ and $f_4$ minima:

In [ ]:
f4_argmin = np.argmin(f4_dvf)
f8_argmin = np.argmin(f8_dvf)

print(f"f4's range: ({np.min(f4_dvf):.2f} — {np.max(f4_dvf):.2f}) → {np.ptp(f4_dvf):.3f}")
print(f"Minimum of f4: {np.min(f4_dvf):.2f}")
print(f"f4 evaluated at f8's optimun location: {f4_dvf[argmin]:.2f}")
delt = f4_dvf[argmin] - np.min(f4_dvf)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f4_dvf)*100:.2f}% of full range")
print()
print(f"f8's range: ({np.min(f8_dvf):.2f} — {np.max(f8_dvf):.2f}) → {np.ptp(f8_dvf):.3f}")
print(f"Minimum of f8: {np.min(f8_dvf):.2f}")
print(f"f8 evaluated at f4's optimun location: {f8_dvf[np.unravel_index(f4_argmin, f4_dvf.shape)]:.2f}")
delt = f8_dvf[np.unravel_index(f4_argmin, f4_dvf.shape)] - np.min(f8_dvf)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f8_dvf)*100:.2f}% of full range")


print("\n Beam sizes at DVF position:")

print(f"\nAt f4's minimum: \n \t {df_at_dvf['sigma_x'][f4_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f4_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_dvf['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f8_argmin]*1000:.2f} μm")

print("\n Beam sizes at focus position:")

print(f"\nAt f4's minimum: \n \t {df_at_focus['sigma_x'][f4_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f4_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_focus['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f8_argmin]*1000:.2f} μm")


print(f"\n Astigmatism at f4's minimum: {df_at_dvf['astigmatism'][f4_argmin]:.2f} mm")

This means that using $f_4$ is a good way of approaching a region not far from the more comprehensive
minimum while only capturing information at a fixed DVF position and only looking at beam data as
projected onto the DVF (beam size and intensity as defined above). This seems a reasonable approach
to online optimization of the CAX beamline. We actually get beam size `19.92 μm` by `11.52 μm` at 
focus position, which match the predicted values for beam size! It appears from the 
check we've just performed that optimizing the beam as seen on the DVF also results in an optimization 
of the beam at the focus position.

Now we will take a look at what happens if the acquisition is already taken in the focus position 
$\bar f$ as defined above. 

If we re-define our functionals to be measuring the observables at the focus position $\bar f$, we 
may expect to see a natural convergence of objectives: as focus matching occurs the size and intensity
measurements taken at $\bar f$ tend to naturally converge to their optimum values. Again, this kind 
of optimization procedure is, at the moment, impractical (or rather impossible!) to perform at the 
real beamline. 

In [ ]:
n = 81

tx_vals = np.asarray(df_at_focus['TX']).reshape(n, n)
ry_vals = np.asarray(df_at_focus['RY']).reshape(n, n)

intensity_vals = np.asarray(df_at_focus['Intensity']).reshape(n, n)
sigmax_vals = np.asarray(df_at_focus['sigma_x']).reshape(n, n)
sigmay_vals = np.asarray(df_at_focus['sigma_y']).reshape(n, n)

astig_vals = np.asarray(df_at_focus['astigmatism']).reshape(n, n)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(17, 11))

axs = axs.flatten()

# title
fig.suptitle("Observables at Focus ($z=\\bar f$)", fontsize=16)

obs = [-intensity_vals, np.abs(astig_vals), 1000*sigmax_vals, 1000*sigmay_vals]
obs_lbl = ["$-I$ [count]", "$|d|$ [mm]", "$\\sigma_x$ [$\\mu$ m]", "$\\sigma_y$ [$\\mu$ m]"]
obs_names = ["negative Intensity", "Astigmatism", "Sigma x", "Sigma y"]

for i in range(4):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(tx_vals, ry_vals, obs[i], levels=15, colors='white', linewidths=0.2)
    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], legend=False)

    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Minimum at ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



The behavior of observables is similar to the prior case, and astigmatism information is in fact the
exact same, as it is a global measure independent of DVF placement. Onwards to the normalized data:

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(17, 11))

axs = axs.flatten()

fig.suptitle("Normalized Observables at focus ($z=\\bar f$)", fontsize=16)


obs = [-intensity_vals/I0, np.abs(astig_vals)/Delta, 1000*sigmax_vals/sigmax_0, 1000*sigmay_vals/sigmay_0]
obs_lbl = ["$-I/I_0$", "$|d|/\\Delta$", "$\\sigma_x/\\sigma_x^0$", "$\\sigma_y/\\sigma_y^0$"]
obs_names = ["normalized Intensity", "normalized Astigmatism", "normalized Sigma x", "normalized Sigma y"]

for i in range(4):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(tx_vals, ry_vals, obs[i], levels=15, colors='white', linewidths=0.2)

    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], legend=False)

    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Minimum at ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



Now we re-define the 8 objectives and plot them:

In [ ]:
f1_focus = (1000)**2*sigmax_vals*sigmay_vals/intensity_vals
f2_focus = 1000*sigmax_vals/sigmax_0 + 1000*sigmay_vals/sigmay_0
f3_focus = -intensity_vals/I0
f4_focus = f2_focus + f3_focus
f5_focus = np.abs(astig_vals)/Delta
f6_focus = f2_focus + f5_focus
f7_focus = f3_focus + f5_focus
f8_focus = f4_focus + f5_focus

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(17, 24))

axs = axs.flatten()

fig.suptitle("Objectives at Focus ($z=\\bar f$)", fontsize=16)

obs = [f1_focus, f2_focus, f3_focus, f4_focus, f5_focus, f6_focus, f7_focus, f8_focus]
obs_lbl = ["$\\sigma_x\\sigma_y/I$", 
           "$\\sigma_x/\\sigma_x^0 + \\sigma_y/\\sigma_y^0$", 
           "$-I/I_0$", 
           "$f_2(\\sigma_x, \\sigma_y) + f_3(I)$",
           "$|d|/\\Delta$",
           "$f_2(\\sigma_x, \\sigma_y) + f_5(d)$",
           "$f_3(I) + f_5(d)$",
           "$f_4(\\sigma_x, \\sigma_y, I) + f_5(d)$"]
obs_names = ["Mixed geometric intensity $f_1(\\sigma_x, \\sigma_y, I)$", 
             "Geometric $f_2(\\sigma_x, \\sigma_y)$", 
             "Intensity $f_3(I)$",
             "Geometric and intensity $f_4(\\sigma_x, \\sigma_y, I)$", 
             "Astigmatism $f_5(d)$",
             "Geometric and astigmatism $f_6(\\sigma_x, \\sigma_y, d)$",
             "Intensity and astigmatism $f_7(I, d)$",
             "Geometric, intensity and astigmatism $f_8(\\sigma_x, \\sigma_y, I, d)$"]

for i in range(8):

    p = axs[i].pcolormesh(tx_vals, ry_vals, obs[i], shading='auto')
    argmin = np.unravel_index(obs[i].argmin(), obs[i].shape)

    tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

    axs[i].scatter(tx_min, ry_min, c='white', marker='d',s=10)
    axs[i].contour(tx_vals, ry_vals, obs[i], levels=15, colors='white', linewidths=0.2)

    add_minimum_band_legend(axs[i], tx_vals, ry_vals, obs[i], legend=False)

    axs[i].set_xlabel("TX")
    axs[i].set_ylabel("RY")
    axs[i].set_title(obs_names[i] + 
                     f" Min @ ({round(tx_min, 2)}, {ry_min:.3f}) = {obs[i][argmin]:.2f}")
    fig.colorbar(p, ax=axs[i], label=obs_lbl[i])



An interesting hehaviour emerges: the geometric optimal location dominates the intensity optimal and
astigmatism optimal dominates all others, but these two distinct minima are very close. Let's compare
the beam sizes at the focus and DVF positions for these two conditions (i.e $f_4$ and $f_8$):

In [ ]:
f4_argmin = np.argmin(f4_focus)
f8_argmin = np.argmin(f8_focus)

argmin = np.unravel_index(f8_focus.argmin(), f8_focus.shape)

print(f"f4's range: ({np.min(f4_focus):.2f} — {np.max(f4_focus):.2f}) → {np.ptp(f4_focus):.3f}")
print(f"Minimum of f4: {np.min(f4_focus):.2f}")
print(f"f4 evaluated at f8's optimun location: {f4_focus[argmin]:.2f}")
delt = f4_focus[argmin] - np.min(f4_focus)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f4_focus)*100:.2f}% of full range")
print()
print(f"f8's range: ({np.min(f8_focus):.2f} — {np.max(f8_focus):.2f}) → {np.ptp(f8_focus):.3f}")
print(f"Minimum of f8: {np.min(f8_focus):.2f}")
print(f"f8 evaluated at f4's optimun location: {f8_focus[np.unravel_index(f4_argmin, f4_focus.shape)]:.2f}")
delt = f8_focus[np.unravel_index(f4_argmin, f4_focus.shape)] - np.min(f8_focus)
print(f"Delta from the optimum: {delt:.2f} = {delt/np.ptp(f8_focus)*100:.2f}% of full range")


print("\n Beam sizes at DVF position:")

print(f"\nAt f4's minimum: \n \t {df_at_dvf['sigma_x'][f4_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f4_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_dvf['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_dvf['sigma_y'][f8_argmin]*1000:.2f} μm")

print("\n Beam sizes at focus position:")

print(f"\nAt f4's minimum: \n \t {df_at_focus['sigma_x'][f4_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f4_argmin]*1000:.2f} μm")
print(f"At f8's minimum: \n \t {df_at_focus['sigma_x'][f8_argmin]*1000:.2f} μm X {df_at_focus['sigma_y'][f8_argmin]*1000:.2f} μm")


print(f"\n Astigmatism at f4's minimum: {df_at_dvf['astigmatism'][f4_argmin]:.2f} mm")

As expected, since the information is already taken at the focus position, every observable converges
together to the true system minimum. The beam sizes at focus are `19.75 μm` by `11.37 μm` which are in
the range of expected values and of comparable size to the results obtained when fixing the DVF position.
The optimal points in knob-space are essentially the same, which is a solid indication of that particular
region being the global minimum of the system.

What can we conclude from this? Limiting ourselves to acquire data at $z=0$ and without DVF scans can 
lead to optimization the beam sizes at the focus position and can improve the foci matching, although
it isn't perfected. 

## RCDS optimizer application

The question now is weather the RCDS optimizer can find this minimum, as it took some good 9 hours
to construct the grid shown above. We will also try to optimize all 8 functions to see the optimizer's
performance. 

Again, the ideal case would be to find one of the minima revealed above with as little information as 
needed, ideally only moving the motors and recording beam properties as seen on the DVF (optimizing $f_4$).

In [ ]:
from apsuite.optimization.rcds import RCDSParams, RCDS

Good, now we can define the functions to evaluate each observable independently and combine them to 
generate the 8 cost functions:

In [ ]:
# Elementary acquisitions
def sigma_x_acq(image: Image):
    "Image sigma x in μm"
    ana = ana_from_image(image)

    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)
    return ana.hprm_fitting['sig_major']


def sigma_y_acq(image: Image):
    "Image sigma y in μm"
    ana = ana_from_image(image)

    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)
    return ana.hprm_fitting['sig_minor']

def astigmatism_acq(cax: CAXSim):
    "Astigmatism acquisition"
    _, _, _, _, _, _, astig, _ = _astigmatism_acquisition(cax)

    return astig

In [ ]:
# Separate objectives at DVF position
def f_1(cax: CAXSim):

    img = cax.dvf_B1.image

    return sigma_x_acq(img)*sigma_y_acq(img)/peak(img)

def f_2(cax: CAXSim):

    img = cax.dvf_B1.image

    return sigma_x_acq(img)/sigmax_0 + sigma_y_acq(img)/sigmay_0

def f_3(cax: CAXSim):

    img = cax.dvf_B1.image

    return - peak(img)/I0

def f_4(cax: CAXSim):

    return f_2(cax) + f_3(cax)

def f_5(cax: CAXSim):

    return np.abs(astigmatism_acq(cax))/Delta

def f_6(cax: CAXSim):

    return f_2(cax) + f_5(cax)

def f_7(cax: CAXSim):

    return f_3(cax) + f_5(cax)

def f_8(cax: CAXSim):

    return f_4(cax) + f_5(cax)
    

Let's define RCDS parameters and try to optimize $f_4$, changing the parameters to achieve convergence
with as few evaluations as possible:

In [ ]:
cax = CAXSim(total_rays=500000, image_size=(100, 100))

argmi = np.unravel_index(f1_dvf.argmin(), f1_dvf.shape)
cax.mirror.tx = tx_vals[argmi]
cax.mirror.ry = ry_vals[argmi]

cax.trace()

f_1(cax), f1_dvf.min()

In [ ]:
cax.mirror.tx = 2.37392681
cax.mirror.ry = 0.09998821

In [ ]:
f_4(cax)

In [ ]:
params = RCDSParams()

In [ ]:
# motor limits: TX and RY
params.limit_lower = [np.min(tx_vals)/2, np.min(ry_vals)] # -4 mm and -0.1 mrad and -15cm
params.limit_upper = [np.max(tx_vals)/2, np.max(ry_vals)] # 4 mm and 0.1 mrad and 15cm
params.caxsim = CAXSim(total_rays=500000, image_size=(100, 100))
# RCDS params
# these are the parameters RCDS requires for its internal
# logic and operations
params.max_number_evals = 500
params.max_number_iters = 12
params.initial_stepsize = 0.5
params.noise_level = 0.01
params.tolerance = 1e-7
params.update_search_directions = True
params.update_search_directions = True

# initial position refers to the initial point in the
# optimaztion variables space
# you could set it to a particular value or sample
# the available search space ramdomly, as below
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
params.initial_position = np.random.uniform(
    low=params.limit_lower, high=params.limit_upper)
# params.initial_position = np.array([-3.6, 0.09])
# search directions along whith to start the line
# minimizations. It is a matrix with each row standing for a
# given direction

# initial directions are purely TX and RY
params.initial_search_directions = np.eye(2)
# params.initial_search_directions = np.array([dir2, dir1])

print(params)

In [ ]:
class CAXRCDS(RCDS):

    def __init__(self, use_thread=True, isonline=True):
        super().__init__(use_thread=use_thread, isonline=isonline)
        self.params = params

    def objective_function(self, pos):

        tx, ry = pos
        caxsim = self.params.caxsim

        caxsim.mirror.tx = tx
        caxsim.mirror.ry = ry

        caxsim.trace()

        # return -f_4(caxsim)**(-9)
        return f_1(caxsim)

In [ ]:
for n in range(2, 10):

    f4_n = -f4_dvf**(-n)

    plt.pcolormesh(tx_vals, ry_vals, f4_n, shading='auto')
    plt.colorbar(label=f"$f_4^{{1/{n}}}$")
    plt.title(f"Objective function $f_4^{{1/{n}}}$ at DVF ($z=0$)")
    levels = np.linspace(np.min(f4_n), np.max(f4_n), 15)
    plt.contour(tx_vals, ry_vals, f4_n, levels=levels, colors='white', linewidths=0.2)
    argmin = np.unravel_index(f4_n.argmin(), f4_n.shape)
    plt.xlabel("TX")
    plt.ylabel("RY")
    plt.scatter(tx_vals[argmin], ry_vals[argmin], c='white', marker='d', s=10, label=f"Minimum at ({tx_vals[argmin]:.2f}, {ry_vals[argmin]:.3f}) = {f4_n[argmin]:.2f}")
    plt.legend()
    plt.show()

plt.pcolormesh(tx_vals, ry_vals, np.log(f4_dvf), shading='auto')
plt.colorbar(label=f"$f_4^{{1/{n}}}$")
plt.title(f"Objective function $f_4^{{1/{n}}}$ at DVF ($z=0$)")
levels = np.linspace(np.min(np.log(f4_dvf)), np.max(np.log(f4_dvf)), 15)
plt.contour(tx_vals, ry_vals, np.log(f4_dvf), levels=levels, colors='white', linewidths=0.2)
argmin = np.unravel_index(np.log(f4_dvf).argmin(), np.log(f4_dvf).shape)
plt.xlabel("TX")
plt.ylabel("RY")
plt.scatter(tx_vals[argmin], ry_vals[argmin], c='white', marker='d', s=10, label=f"Minimum at ({tx_vals[argmin]:.2f}, {ry_vals[argmin]:.3f}) = {np.log(f4_dvf)[argmin]:.2f}")
plt.legend()
plt.show()

In [ ]:
opt = CAXRCDS(isonline=False, use_thread=False)

In [ ]:
opt.start()

In [ ]:
opt.plot_history()
plt.show()

In [ ]:
cax = CAXSim();

In [ ]:
print("Optimization Overview \n" + "="*50)

print(f"{'Evaluations':<20}: {opt.num_objective_evals}")
print()
print(f"{'Initial objective':<20}: {opt.objfuncs_evaluated[0]:.4f}")
print(f"{'Best objective':<20}: {min(opt.objfuncs_best):.4f}")
print(f"{'f1 array minimum':<20}: {np.min(f1_dvf):.4f}")
print()
print(f"{'Initial position':<20}: {opt.positions_evaluated[0]}")
print(f"{'Best position':<20}: {opt.positions_best[np.argmin(opt.objfuncs_best)]}")
argmin = np.unravel_index(np.argmin(f1_dvf), f1_dvf.shape)
print(f"{'f1 position':<20}: [{tx_vals[argmin]:.8f} {ry_vals[argmin]:.8f}]")


In [ ]:
opt.objfuncs_best[-1], min(opt.objfuncs_best)

In [ ]:
opt.positions_best[np.argmin(opt.objfuncs_best)]

In [ ]:
fig, ax = opt.plot_knobspace_slice(dir_idcs=(1, 2))
fig.set_size_inches(10, 8)
# 2. Create the X and Y grid coordinates

# Filled contour (the colormap background)
# filled_contour = ax.pcolormesh(TX, RY, f4_dvf, cmap='viridis')
p = ax.pcolormesh(tx_vals, ry_vals, f1_dvf, shading='auto')
argmin = np.unravel_index(f1_dvf.argmin(), f1_dvf.shape)

tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

ax.scatter(tx_min, ry_min, c='white', marker='d',s=15)

# Contour lines over the top (optional, for crisp boundaries)

# 0.5% above minimum filled contour
obj_amplitude = np.ptp(f1_dvf)
level = np.min(f1_dvf) + 0.01 * obj_amplitude
levels = np.linspace(level, np.max(f1_dvf), 20)
lines = ax.contour(tx_vals, ry_vals, f1_dvf, levels=levels, colors='white', linewidths=0.2)
add_minimum_band_legend(ax, tx_vals, ry_vals, f1_dvf, fraction=0.01, legend=True)

# Label the contour lines with their actual data values
# plt.clabel(lines, inline=True, fontsize=8)

# Add a colorbar legend
plt.colorbar(p, label='Function Value Z')

# Formatting labels
plt.title('TX — RY knobspace slice with f4 objective function', fontsize=14)
plt.xlabel('TX')
plt.ylabel('RY')
plt.show()
# fig


In [ ]:
dir1 = [0.5, 0.025]
dir2 = [0.5, -0.0125]

plt.arrow(0, 0, dir1[0], dir1[1], color='red', head_width=0.02, head_length=0.02, label='Direction 1')
plt.arrow(0, 0, dir2[0], dir2[1], color='blue', head_width=0.02, head_length=0.02, label='Direction 2')
plt.xlim(-4, 4)
plt.ylim(-0.1, 0.1)

In [ ]:
cax = CAXSim(total_rays=int(2e5))

tx_i, ry_i = opt.params.initial_position
cax.mirror.tx = tx_i
cax.mirror.ry = ry_i

img = Image2D_Fit(data=cax.dvf_B1.image.histogram.T, use_svd4theta=True)

display_image(cax.dvf_B1.image)
plt.show()

sig_x = img.fitx.roi_sigma
Del_x = np.diff(cax.dvf_B1.image.bin_h_edges)[0]*1000   # Δx in μm
sig_x *= Del_x

sig_y = img.fity.roi_sigma
Del_y = np.diff(cax.dvf_B1.image.bin_v_edges)[0]*1000   # Δy in μm
sig_y *= Del_y

sig_xi, sig_yi = sig_x, sig_y



tx_s, ry_s = opt.positions_best[np.argmin(opt.objfuncs_best)]

cax.mirror.tx = tx_s
cax.mirror.ry = ry_s

display_image(cax.dvf_B1.image)
plt.show()

img = Image2D_Fit(data=cax.dvf_B1.image.histogram.T, use_svd4theta=True)

sig_x = img.fitx.roi_sigma
Del_x = np.diff(cax.dvf_B1.image.bin_h_edges)[0]*1000   # Δx in μm
sig_x *= Del_x

sig_y = img.fity.roi_sigma
Del_y = np.diff(cax.dvf_B1.image.bin_v_edges)[0]*1000   # Δy in μm
sig_y *= Del_y

sig_xf, sig_yf = sig_x, sig_y

print(f"\nInitial position: ({tx_i:.4f}, {ry_i:.4f})")

print(f"Sigma x at DVF : {sig_xi:.2f} μm")
print(f"Sigma y at DVF : {sig_yi:.2f} μm")

print(f"\nBest position: ({tx_s:.4f}, {ry_s:.4f})")

print(f"Sigma x at DVF: {sig_xf:.2f} μm")
print(f"Sigma y at DVF: {sig_yf:.2f} μm")

In [ ]:
fig, ax = opt.plot_knobspace_slice(dir_idcs=(1, 2))
fig.set_size_inches(10, 8)
# 2. Create the X and Y grid coordinates

# Filled contour (the colormap background)
filled_contour = ax.pcolormesh(tx_vals, ry_vals, f8_dvf, cmap='viridis')
p = ax.pcolormesh(tx_vals, ry_vals, f8_dvf, shading='auto')
argmin = np.unravel_index(f8_dvf.argmin(), f8_dvf.shape)

tx_min, ry_min = tx_vals[argmin], ry_vals[argmin]

diamond = ax.scatter(tx_min, ry_min, c='white', marker='d',s=15, label='sampled minimum')

# Contour lines over the top (optional, for crisp boundaries)

# 0.5% above minimum filled contour
obj_amplitude = np.ptp(f8_dvf)
level = np.min(f8_dvf) + 0.005 * obj_amplitude
levels = np.linspace(level, np.max(f8_dvf), 20)
lines = ax.contour(tx_vals, ry_vals, f8_dvf, levels=levels, colors='white', linewidths=0.2)
add_minimum_band_legend(ax, tx_vals, ry_vals, f8_dvf, legend=True)


# ax.legend(loc='upper right', fontsize=10, frameon=True, facecolor='white', edgecolor='black')
# # Label the contour lines with their actual data values
# # plt.clabel(lines, inline=True, fontsize=8)
# proxy = plt.Rectangle((0,0),1,1,fc = cf.collections[0].get_facecolor()[0], label='0.5% above minimum')
# # Add a colorbar legend
# # plt.colorbar(filled_contour, label='Function Value Z')
# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=handles+[proxy], loc='lower right', fontsize=10, frameon=True, edgecolor='black')
# Formatting labels
# plt.title('2D Function Contour and Colormap')
# plt.xlabel('X-axis')
# plt.ylabel('Y-axis')
# fig


⚠️ As mentioned, by switching the analysis method (currently getting the raw data from `Shadow`), the 
simulation results are underestimating beam sizes, which explains the facth that the optimizer can reach
a final objective *lower* than the minima we found from the grid data.

This means the optimizer's performance is artifiically enhanced in the results presented here. In any case,
the conclusion at this point is that the optimizer parameters used here are enough to always reach the 0.5%
margin above the minimum consistently, it now remains to re-introduce the better analysis methods and 
validate these conclusions once again.

### New re-acquisitions

In [ ]:
tx_vals_flat = tx_vals[:, 0]/2
ry_vals_flat = ry_vals[0, :]

In [ ]:
tx_vals_flat

In [ ]:
records_dvf = []

for tx in tx_vals_flat:
    for ry in ry_vals_flat:

        cax.mirror.tx = tx
        cax.mirror.ry = ry

        I_dvf = peak(cax.dvf_B1.image)

        sig_x = sigma_x_acq(cax.dvf_B1.image)
        sig_y = sigma_y_acq(cax.dvf_B1.image)

        record_dvf = {
            'TX'          : tx,
            'RY'          : ry,
            'Intensity'   : I_dvf,
            'sigma_x'     : sig_x,
            'sigma_y'     : sig_y,
        }

        records_dvf.append(record_dvf)


new_df_at_dvf = pd.DataFrame(records_dvf)
new_df_at_dvf.to_csv('20261028_data.csv', index=False)

### Tuning RCDS parameters

In [ ]:
params = RCDSParams()

# motor limits: TX and RY
params.limit_lower = [np.min(tx_vals), np.min(ry_vals)] # -4 mm and -0.1 mrad
params.limit_upper = [np.max(tx_vals), np.max(ry_vals)] # +4 mm and +0.1 mrad
# RCDS params
# these are the parameters RCDS requires for its internal
# logic and operations
params.max_number_evals = 200
params.update_search_directions = True
params.update_search_directions = True

# initial position refers to the initial point in the
# optimaztion variables space
# you could set it to a particular value or sample
# the available search space ramdomly, as below
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
# search directions along whith to start the line
# minimizations. It is a matrix with each row standing for a
# given direction

# initial directions are purely TX and RY
params.initial_search_directions = np.eye(2)
# params.initial_search_directions = np.array([dir2, dir1])

print(params)

In [ ]:
params.caxsim = CAXSim()
params.max_number_iters = 6
params.initial_stepsize = 0.01
params.noise_level = 1e-4
params.tolerance = 1e-7
params.initial_position = np.array([-3.6, 0.09])

In [ ]:
rcds_param_knobs = {     # 3 x 3 x 1 x 1 = 9
    'max_number_iters': [6, 10, 12],
    'initial_stepsize': [1.e-1, 2.e-1, 5.e-1],
    'noise_level'     : [1.e-4],
    'tolerance'       : [1.e-7]
}

In [ ]:
# 25 samples
margin_tx = 0.5 # 0.5 mm
sample_tx = np.linspace(params.limit_lower[0]+margin_tx, params.limit_upper[0]-margin_tx, 3)

margin_ry = 0.01 # 0.01 mrad
sample_ry = np.linspace(params.limit_lower[1]+margin_ry, params.limit_upper[1]-margin_ry, 3)
TXS, RYS = np.meshgrid(sample_tx, sample_ry)
plt.scatter(TXS, RYS, c='red', marker='o', s=10, label='Sampled points')

In [ ]:
class CAXRCDS(RCDS):

    def __init__(self, params, use_thread=True, isonline=True):
        super().__init__(use_thread=use_thread, isonline=isonline)
        self.params = params

    def objective_function(self, pos):

        tx, ry = pos
        caxsim = self.params.caxsim

        caxsim.mirror.tx = tx
        caxsim.mirror.ry = ry

        caxsim.trace()

        return f_1(caxsim)

In [ ]:
from itertools import product


mult_params = list(product(*(sets for sets in rcds_param_knobs.values())))

tuning_params = pd.DataFrame()

mean_best_objfuncs = []
mean_pos_distances = []
mean_reach_1pct    = []
mean_evals         = []
records            = []
argmin = np.unravel_index(np.argmin(f1_dvf), f1_dvf.shape)

for i, param_set in enumerate(mult_params):
    for j, key in enumerate(rcds_param_knobs.keys()):
        setattr(params, key, param_set[j])
        params.caxsim = CAXSim()

    best_objfuncs = []
    pos_distances = []
    reach_1pct    = []
    evals         = []
    
    for tx in sample_tx:
        for ry in sample_ry:

            params.initial_position = np.array([tx, ry])

            opt = CAXRCDS(params=params, use_thread=False, isonline=False)

            opt.start()

            best_objfuncs.append(min(opt.objfuncs_best))

            pos_distances.append(np.linalg.norm(opt.positions_best[np.argmin(opt.objfuncs_best)] - np.array([tx_vals[argmin], ry_vals[argmin]])))
            reached_1pct = int(np.min(opt.objfuncs_best) <= np.min(f1_dvf) + 0.01 * np.ptp(f1_dvf))
            evals.append(opt.num_objective_evals)
    mean_best_objfuncs.append(np.mean(best_objfuncs))
    mean_pos_distances.append(np.mean(pos_distances))
    mean_reach_1pct.append(np.mean(reached_1pct))
    mean_evals.append(np.mean(evals))
    print(f"Run {i+1}/{len(mult_params)} with params: {param_set}")
    print(f"Best objective: {mean_best_objfuncs[-1]:.4f}")
    print(f"Best position distance to min: {mean_pos_distances[-1]:.4f}")
    print(f"Mean distance to f1 minimum: {mean_pos_distances[-1]:.4f}")
    print(f"Mean evaluations: {mean_evals[-1]:.4f}")

    records.append({
        'max_number_iters': param_set[0],
        'initial_stepsize': param_set[1],
        'noise_level'     : param_set[2],
        'tolerance'       : param_set[3],
        'mean_best_objfuncs': mean_best_objfuncs[-1],
        'mean_pos_distances': mean_pos_distances[-1],
        'mean_evals'       : mean_evals[-1]
    })

tuning_params = pd.DataFrame(records)



In [ ]:
tuning_params.to_csv('rcds_tuning_results.csv', index=False)

#minimum mean_best_objfuncs' parameter set
best_param_set = tuning_params.loc[tuning_params['mean_best_objfuncs'].idxmin()]
best_param_set

In [ ]:
tuning_params

In [ ]:
best_param_set = tuning_params.loc[tuning_params['mean_best_objfuncs'].idxmin()]
best_param_set